In [2]:
import pandas as pd
import torch
import torch.nn as nn 
import numpy as np
import re

from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.optim import AdamW

In [3]:
df = pd.read_csv("/kaggle/input/datasets/rianluisx/dataset/data/processed/2022/cleaned/2022_combined_coarse.csv")

In [4]:
label_mapping = {
    'Negative': 0,
    'Neutral': 1,
    'Positive': 2
}
df['label'] = df.sentiment.map(label_mapping)

df = df.dropna(subset=['text', 'label'])
df['label'] = df['label'].astype(int)

train_df, val_df = train_test_split(df, test_size=0.2, random_state=42)

tokenizer = AutoTokenizer.from_pretrained("bert-base-multilingual-uncased")

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [5]:
class TweetDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len = 128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, index):
        text = str(self.texts[index])
        label = self.labels[index]

        encoding = self.tokenizer(
            text,
            max_length = self.max_len,
            padding = 'max_length',
            truncation = True,
            return_tensors = 'pt'
        )

        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'label': torch.tensor(label, dtype=torch.long)
        }

In [6]:
train_dataset = TweetDataset(train_df['text'].values, train_df['label'].values, tokenizer)
val_dataset = TweetDataset(val_df['text'].values, val_df['label'].values, tokenizer)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on: {device}")
if torch.cuda.device_count() > 1:
    print(f"Multi-GPU activated! Using {torch.cuda.device_count()} GPUs.")

model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-multilingual-uncased",
    num_labels = 3
)

if torch.cuda.device_count() > 1:
    model = nn.DataParallel(model)

model.to(device)

optimizer = AdamW(model.parameters(), lr=2e-5)
scaler = torch.amp.GradScaler('cuda')

Training on: cuda
Multi-GPU activated! Using 2 GPUs.


model.safetensors:   0%|          | 0.00/672M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-multilingual-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [7]:
epochs = 3

for epoch in range(epochs):

    model.train()
    total_train_loss = 0

    for batch in train_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)

        optimizer.zero_grad()

        with torch.autocast(device_type='cuda'):
            outputs = model(
                input_ids = input_ids,
                attention_mask = attention_mask,
                labels = labels
            )
            loss = outputs.loss.mean() if torch.cuda.device_count() > 1 else outputs.loss

        total_train_loss += loss.item()

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

    avg_train_loss = total_train_loss / len(train_loader)


    model.eval()
    total_val_loss = 0
    
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["label"].to(device)

            with torch.autocast(device_type = 'cuda'):
                outputs = model(
                    input_ids = input_ids,
                    attention_mask = attention_mask,
                    labels = labels
                )
                loss = outputs.loss.mean() if torch.cuda.device_count() > 1 else outputs.loss
                total_val_loss += loss.item()

                logits = outputs.logits
                preds = torch.argmax(logits, dim=1)
                
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())

    avg_val_loss = total_val_loss / len(val_loader)


    acc = accuracy_score(all_labels, all_preds)
    precision, recall, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average='weighted', zero_division=0)

    print(
        f"Epoch {epoch+1}/{epochs} | "
        f"Train Loss: {avg_train_loss:.4f} | "
        f"Val Loss: {avg_val_loss:.4f} | "
        f"Acc: {acc:.4f} | "
        f"F1: {f1:.4f} | "
        f"Prec: {precision:.4f} | "
        f"Rec: {recall:.4f}"
    )

print("\n--- Training Complete ---")


print("\nFinal Epoch Classification Report:")
target_names = ['Negative', 'Neutral', 'Positive']
print(classification_report(all_labels, all_preds, target_names=target_names, zero_division=0))


weights_path = "./mbert_kaggle_weights.pth"
tokenizer_path = "./saved_tokenizer"


model_to_save = model.module if hasattr(model, 'module') else model


torch.save(model_to_save.state_dict(), weights_path)


tokenizer.save_pretrained(tokenizer_path)

print(f"\nModel weights successfully saved to: {weights_path}")
print(f"Tokenizer successfully saved to: {tokenizer_path}")

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch 1/3 | Train Loss: 0.7968 | Val Loss: 0.6883 | Acc: 0.6924 | F1: 0.6868 | Prec: 0.6890 | Rec: 0.6924


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch 2/3 | Train Loss: 0.6515 | Val Loss: 0.6638 | Acc: 0.7017 | F1: 0.6853 | Prec: 0.7028 | Rec: 0.7017


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch 3/3 | Train Loss: 0.5399 | Val Loss: 0.7062 | Acc: 0.7105 | F1: 0.7028 | Prec: 0.7049 | Rec: 0.7105

--- Training Complete ---

Final Epoch Classification Report:
              precision    recall  f1-score   support

    Negative       0.77      0.87      0.82      1681
     Neutral       0.68      0.72      0.70      1780
    Positive       0.66      0.48      0.56      1279

    accuracy                           0.71      4740
   macro avg       0.70      0.69      0.69      4740
weighted avg       0.70      0.71      0.70      4740


Model weights successfully saved to: ./mbert_kaggle_weights.pth
Tokenizer successfully saved to: ./saved_tokenizer


In [8]:
def preprocess_text(text):
    if not isinstance(text, str):
        return ""
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)
    text = re.sub(r'@\S+', '', text)
    text = text.replace('#', '') 
    
    SENATORS_2025 = [
        "benhur abalos", "jovilyn aceron", "kuya aclan", "mercedita acopiado",
        "jerome adonis", "abel adorable", "dingbicol advincula", "peter joemel advincula",
        "robert agad", "cez aguilar", "eric alcantara", "wilson amad",
        "salipada amir hussin", "nelson ancajas", "nars alyn andamo", "kuya manny andrada",
        "bobie andrino", "joel apolinario", "bam aquino", "primo aquino",
        "ronnel arambulo", "gerald arcega", "ernesto arellano", "jerson ares",
        "ricarda arguelles", "adam artajo", "ismael bajo", "kaka balite",
        "roberto ballon", "jaime balmas", "loreto banosan", "rodolfo basilan",
        "eduardo bautista", "elvis beniga", "dave biazon", "maria charito billones",
        "abby binay", "chito bonayog", "jimmy bondoc", "ramon jr. bong revilla",
        "bonifacio bosita", "warly bovier", "arlene brosas", "hernando bruce",
        "injim bunayog", "jose bunilla", "june king sbc cabalida", "roy cabonegro",
        "leo cadion", "ondong capuno", "allen capuyan", "khaled casimra",
        "teddy casiño", "agapito casipong", "teacher france castro", "mig caturan",
        "berteni causing", "laurel cay", "pia cayetano", "gabriel chaclag",
        "david paul chan", "emilio chan", "raffy chico", "shirly cuatchin",
        "david d'angelo", "de alban attorney angelo", "ka leody de guzman", "orlando de guzman",
        "bato dela rosa", "joseph delgado", "phil delos reyes", "ding diaz",
        "sunang ditanongun", "gem domagtoy", "ka sonny domingo", "nanay mimi doringo",
        "edgardo duque", "joseph dy", "alexander encarnacion", "maria fe era",
        "arnel escobal", "john escobar", "luke espiritu", "idolodi estrella",
        "mody piston floranda", "marc louie gamboa", "jonry gargarita", "bong go go",
        "norberto gonzales", "jb guigayuma", "jayvee hinlo", "gringo honasan",
        "victor inte", "relly jr. jose", "alice jumalon", "bae dalan kokkinaras",
        "ping lacson", "sixto lagare", "alex lague", "raul lambino",
        "princedatu lamoste", "lito lapid", "manoy wilbert lee", "amirah lidasan",
        "beth lopez", "happy lubarbio", "melchor lucañas", "romeo macaraeg",
        "daniel magtira", "fredie maiquez", "getter malinao", "magno manalo",
        "artemio maquiso", "rodante marcoleta", "francis leo marcos", "imee r. marcos",
        "marcos tallano tagean", "norman marquez", "eric martinez", "doc marites mata",
        "atty. sonny matula", "liza maza", "heidi mendoza", "ka martin mendoza",
        "luther meniano", "ligtas montealto", "joey montemayor", "subair mustapha",
        "eric negapatan", "richard nicolas", "rex noel", "jose jessie olivar",
        "henry olonan", "doc willie ong", "oscar ongjoco", "pete ordiales",
        "manny pacman pacquiao", "roel pacquiao", "janice padilla", "super mar pagaragan",
        "nheling paliza", "diego jr. palomares", "kiko pangilinan", "tony par",
        "eulogio jr. partosa", "epifanio perez", "bong bong pimentel", "rastaman plaza",
        "ariel porfirio querubin", "apollo quiboloy", "princess jade ramos", "danilo ramos",
        "kuya wily red", "randy red", "randy restum", "willie wil revillame",
        "james reyes", "willie ricablanca", "atty. vic rodriguez", "elpidio jr. rosales",
        "edmon rubi", "virginia sabit", "nur-ana sahidulla", "jimmy salapantan",
        "najar salih", "phillip ipe salvador", "mulo san ramon", "melissa sanchez",
        "roberto sembrano", "froilan serafico", "manong chavit singson", "tito sotto",
        "joey tam", "michael bongbong tapado", "francis tolentino", "omar tomanong",
        "ben bitag tulfo", "erwin tulfo", "ferdinand tuzara", "faith ugsad",
        "mar manibela valbuena", "leandro verceles", "camille villar"
    ]

    search = "leni robredo bongbong marcos isko moreno domagoso manny pacman pacquiao ping lacson ernie abella leody de guzman norberto gonzales jose montemayor jr faisal mangondato"
    CANDIDATES_LIST_2022 = search.split(" ")
    
    PARTYLISTS_2025 = [] 
    
    CANDIDATES_LIST = " ".join(SENATORS_2025 + PARTYLISTS_2025 + CANDIDATES_LIST_2022).split(" ")
    
    text = text.lower().strip()
    text = re.sub(rf"\b({'|'.join(re.escape(c.lower()) for c in CANDIDATES_LIST)})\b", "candidate", text)
    
    return text

df_2025 = pd.read_csv("/kaggle/input/datasets/rianluisx/dataset/data/raw/2025.csv")
df_2025['text'] = df_2025['text'].apply(preprocess_text)
df_2025 = df_2025.dropna(subset=['text'])
df_2025 = df_2025[df_2025['text'] != '']

texts_2025 = df_2025['text'].values

In [9]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"running on: {device}")

running on: cuda


In [10]:
pseudotokenizer = AutoTokenizer.from_pretrained("bert-base-multilingual-uncased")

pseudomodel = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-multilingual-uncased",
    num_labels=3
)

pseudomodel.load_state_dict(torch.load("/kaggle/working/mbert_kaggle_weights.pth"))

if torch.cuda.device_count() > 1:
    pseudomodel = nn.DataParallel(pseudomodel)

pseudomodel.to(device)
pseudomodel.eval()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-multilingual-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


DataParallel(
  (module): BertForSequenceClassification(
    (bert): BertModel(
      (embeddings): BertEmbeddings(
        (word_embeddings): Embedding(105879, 768, padding_idx=0)
        (position_embeddings): Embedding(512, 768)
        (token_type_embeddings): Embedding(2, 768)
        (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (encoder): BertEncoder(
        (layer): ModuleList(
          (0-11): 12 x BertLayer(
            (attention): BertAttention(
              (self): BertSelfAttention(
                (query): Linear(in_features=768, out_features=768, bias=True)
                (key): Linear(in_features=768, out_features=768, bias=True)
                (value): Linear(in_features=768, out_features=768, bias=True)
                (dropout): Dropout(p=0.1, inplace=False)
              )
              (output): BertSelfOutput(
                (dense): Linear(in_features=768, out_features=768,

In [11]:
class UnlabeledTweetDataset(Dataset):
    def __init__(self, texts, tokenizer, max_len=128):
        self.texts = texts
        self.tokenizer = tokenizer 
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, index):
        text = str(self.texts[index])
        encoding = self.tokenizer(
            text,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten()
        }

inference_dataset = UnlabeledTweetDataset(texts_2025, tokenizer)
inference_loader = DataLoader(inference_dataset, batch_size=32, shuffle=False, num_workers=2)

In [12]:


all_predictions = []
all_confidences = [] 

with torch.no_grad():
    for batch in inference_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        
        with torch.autocast(device_type='cuda'):
            outputs = pseudomodel(input_ids=input_ids, attention_mask=attention_mask)
            probs = torch.nn.functional.softmax(outputs.logits, dim=1)
            confidences, preds = torch.max(probs, dim=1)
            
            all_predictions.extend(preds.cpu().numpy())
            all_confidences.extend(confidences.cpu().numpy())

In [13]:

df_2025['label'] = all_predictions
df_2025['confidence'] = all_confidences
df_2025['sentiment'] = df_2025['label'].map({0: 'Negative', 1: 'Neutral', 2: 'Positive'})

high_quality_df = df_2025[df_2025['confidence'] >= 0.80].copy()


full_dataset_df = high_quality_df.sample(frac=1, random_state=42).reset_index(drop=True)

print(full_dataset_df['sentiment'].value_counts())
print(f"\nTotal rows ready for 2025 training: {len(full_dataset_df)}")

full_dataset_df.to_csv("./2025_pseudolabeled.csv", index=False)

sentiment
Negative    75277
Neutral     37126
Positive     1888
Name: count, dtype: int64

Total rows ready for 2025 training: 114291


In [15]:
pseudolabeled_2025 = pd.read_csv("/kaggle/working/2025_pseudolabeled.csv")
pseudolabeled_2025 = pseudolabeled_2025.dropna(subset=['text', 'label'])
pseudolabeled_2025['label'] = pseudolabeled_2025['label'].astype(int)

In [16]:
student_train_df, student_val_df = train_test_split(pseudolabeled_2025, test_size=0.2, random_state=42)

In [17]:
student_tokenizer = AutoTokenizer.from_pretrained("bert-base-multilingual-uncased")

In [18]:
class PseudoLabeledTweetDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, index):
        encoding = self.tokenizer(
            str(self.texts[index]),
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'label': torch.tensor(self.labels[index], dtype=torch.long)
        }


student_train_dataset = PseudoLabeledTweetDataset(student_train_df['text'].values, student_train_df['label'].values, student_tokenizer)
student_val_dataset = PseudoLabeledTweetDataset(student_val_df['text'].values, student_val_df['label'].values, student_tokenizer)

student_train_loader = DataLoader(student_train_dataset, batch_size=64, shuffle=True, num_workers=2)
student_val_loader = DataLoader(student_val_dataset, batch_size=64, shuffle=False, num_workers=2)

In [19]:
device_student = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"running on: {device_student}")

student_model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-multilingual-uncased",
    num_labels=3
)

if torch.cuda.device_count() > 1:
    student_model = nn.DataParallel(student_model)

student_model.to(device_student)

optimizer_student = AdamW(student_model.parameters(), lr=2e-5)
scaler_student = torch.amp.GradScaler('cuda')

running on: cuda


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-multilingual-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [20]:
student_epochs = 3 
for epoch in range(student_epochs):
    student_model.train()
    total_train_loss = 0
    
    for batch in student_train_loader:
        input_ids = batch['input_ids'].to(device_student)
        attention_mask = batch['attention_mask'].to(device_student)
        labels = batch['label'].to(device_student)

        optimizer_student.zero_grad()

        with torch.autocast(device_type='cuda'):
            outputs = student_model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )
            loss = outputs.loss.mean() if torch.cuda.device_count() > 1 else outputs.loss

        total_train_loss += loss.item()

        scaler_student.scale(loss).backward()
        scaler_student.step(optimizer_student)
        scaler_student.update()

    avg_train_loss = total_train_loss / len(student_train_loader)

    student_model.eval()
    total_val_loss = 0
    student_all_preds = []
    student_all_labels = []

    with torch.no_grad():
    
        for batch in student_val_loader:
            input_ids = batch["input_ids"].to(device_student)
            attention_mask = batch["attention_mask"].to(device_student)
            labels = batch["label"].to(device_student)

            with torch.autocast(device_type='cuda'):
                outputs = student_model(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    labels=labels
                )
                loss = outputs.loss.mean() if torch.cuda.device_count() > 1 else outputs.loss
                total_val_loss += loss.item()

                logits = outputs.logits
                preds = torch.argmax(logits, dim=1)
                
                student_all_preds.extend(preds.cpu().numpy())
                student_all_labels.extend(labels.cpu().numpy())

    avg_val_loss = total_val_loss / len(student_val_loader)

    acc = accuracy_score(student_all_labels, student_all_preds)
    precision, recall, f1, _ = precision_recall_fscore_support(student_all_labels, student_all_preds, average='weighted', zero_division=0)

    print(
        f"Epoch {epoch+1}/{student_epochs} | "
        f"Train Loss: {avg_train_loss:.4f} | "
        f"Val Loss: {avg_val_loss:.4f} | "
        f"Acc: {acc:.4f} | "
        f"F1: {f1:.4f} | "
        f"Prec: {precision:.4f} | "
        f"Rec: {recall:.4f}"
    )

print("Training Complete")

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch 1/3 | Train Loss: 0.0855 | Val Loss: 0.0259 | Acc: 0.9914 | F1: 0.9914 | Prec: 0.9914 | Rec: 0.9914


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch 2/3 | Train Loss: 0.0237 | Val Loss: 0.0235 | Acc: 0.9912 | F1: 0.9912 | Prec: 0.9914 | Rec: 0.9912


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch 3/3 | Train Loss: 0.0139 | Val Loss: 0.0210 | Acc: 0.9934 | F1: 0.9934 | Prec: 0.9935 | Rec: 0.9934
Training Complete


In [22]:
target_names = ['Negative', 'Neutral', 'Positive']

print("\nFinal Epoch Classification Report:")
# Use the newly named student metrics
print(classification_report(student_all_labels, student_all_preds, target_names=target_names, zero_division=0))

final_weights_path = "./mbert_2025_model.pt"

model_to_save = student_model.module if hasattr(student_model, 'module') else student_model

torch.save(model_to_save.state_dict(), final_weights_path)


Final Epoch Classification Report:
              precision    recall  f1-score   support

    Negative       1.00      0.99      1.00     15040
     Neutral       0.99      1.00      0.99      7436
    Positive       0.97      0.95      0.96       383

    accuracy                           0.99     22859
   macro avg       0.99      0.98      0.98     22859
weighted avg       0.99      0.99      0.99     22859



In [24]:
checker = pd.read_csv("/kaggle/working/2025_pseudolabeled.csv")
sentiment_counts = checker['sentiment'].value_counts()
total_count = len(checker)

print("--- Sentiment Breakdown ---")
print(sentiment_counts.to_string())
print("\n--- Total ---")
print(total_count)

--- Sentiment Breakdown ---
sentiment
Negative    75277
Neutral     37126
Positive     1888

--- Total ---
114291
